# 2. Preprocessing

One review followed through every step of `backend/app/nlp/preprocessing.py`, the code used for both training and inference.

In [1]:
import sys
from pathlib import Path

# Make `app` and `training` importable from the notebooks folder.
BACKEND = Path.cwd().resolve().parent / "backend" if Path.cwd().name == "notebooks" else Path.cwd() / "backend"
sys.path.insert(0, str(BACKEND))

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", 120)

## Cleaning and sentence splitting

HTML, odd Unicode and extra whitespace are removed. Words are never removed or stemmed: BERT needs "not" and "but".

In [2]:
from app.nlp.preprocessing import clean_text, split_sentences, tokenize

raw = "  The <b>battery life</b> is great&nbsp;!!   But the touchpad isn\u2019t responsive.\n\nWould not buy again. "
text = clean_text(raw)
print(repr(raw)); print(repr(text))
sentences = split_sentences(text)
pd.DataFrame({"sentence": s.text, "start": s.start, "end": s.end} for s in sentences)

'  The <b>battery life</b> is great&nbsp;!!   But the touchpad isn’t responsive.\n\nWould not buy again. '
'The battery life is great !! But the touchpad isn’t responsive. Would not buy again.'


,sentence,start,end
0,The battery life is great !,0,27
1,!,27,28
2,But the touchpad isn’t responsive.,29,63
3,Would not buy again.,64,84


## Word tokens with character offsets

The offsets map a predicted aspect back to its position in the review, which the interface uses for highlighting.

In [3]:
tokens = tokenize(sentences[0].text, sentences[0].start)
pd.DataFrame({"token": t.text, "start": t.start, "end": t.end} for t in tokens).T

,0,1,2,3,4,5
token,The,battery,life,is,great,!
start,0,4,12,17,20,26
end,3,11,16,19,25,27


## BIO tags

Annotated character spans become one tag per word. `B-ASPECT` starts an aspect, `I-ASPECT` continues it.

In [4]:
from app.nlp.preprocessing import bio_to_spans, spans_to_bio

sentence = "The battery life is great but the screen is dim."
words = tokenize(sentence)
gold = [(4, 16), (34, 40)]            # character spans of "battery life" and "screen"
tags = spans_to_bio(words, gold)
display(pd.DataFrame({"token": [w.text for w in words], "tag": tags}).T)
print("back to token spans (end exclusive):", bio_to_spans(tags))

,0,1,2,3,4,5,6,7,8,9,10
token,The,battery,life,is,great,but,the,screen,is,dim,.
tag,O,B-ASPECT,I-ASPECT,O,O,O,O,B-ASPECT,O,O,O


back to token spans (end exclusive): [(1, 3), (7, 8)]


## Aligning word labels with BERT word-pieces

BERT splits rare words into pieces. Only the first piece of a word keeps the label; the others and the special tokens get -100, which the loss ignores.

In [5]:
from transformers import AutoTokenizer
from training.train_aspect import build_feature

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
record = {"tokens": ["The", "touchpad", "is", "unresponsive"], "tags": ["O", "B-ASPECT", "O", "O"]}
feature = build_feature(tokenizer, record, max_length=128)
pd.DataFrame({"piece": tokenizer.convert_ids_to_tokens(feature["input_ids"]), "label": feature["labels"]}).T

C:\Users\Admin\Desktop\PROJECTS\NLP\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,0,1,2,3,4,5,6,7,8,9
piece,[CLS],the,touch,##pad,is,un,##res,##pon,##sive,[SEP]
label,-100,0,1,-100,0,0,-100,-100,-100,-100


## The two sentiment inputs

The aspect-sentiment model reads the sentence *and* the aspect; the overall model reads the sentence alone.

In [6]:
from app.models.sentiment_model import encode_pair

for aspect in ("battery life", "screen", None):
    ids = encode_pair(tokenizer, sentence, aspect, 128)["input_ids"]
    print(f"{str(aspect):13s}", " ".join(tokenizer.convert_ids_to_tokens(ids)))

battery life  [CLS] the battery life is great but the screen is dim . [SEP] battery life [SEP]
screen        [CLS] the battery life is great but the screen is dim . [SEP] screen [SEP]
None          [CLS] the battery life is great but the screen is dim . [SEP]


## Context and aspect normalisation

The clause shown next to each aspect, and the form used to group aspects in the statistics.

In [7]:
from app.nlp.preprocessing import extract_context, normalize_aspect

print(repr(extract_context(sentence, 4, 16)))
print(repr(extract_context(sentence, 34, 40)))
print({word: normalize_aspect(word) for word in ("Batteries", "battery", "Screens", "battery life")})

'The battery life is great'
'the screen is dim'


{'Batteries': 'battery', 'battery': 'battery', 'Screens': 'screen', 'battery life': 'battery life'}
